In [14]:
import pandas as pd
import numpy as np

## 1. Introdução

### Objetivo da análise


****
## 2. Carregamento e exploração dos dados
Nesta etapa, o conjunto de dados do Titanic é carregado e explorado inicialmente, com o objetivo de compreender sua estrutura, suas variáveis e a presença de possíveis valores ausentes. Essa análise inicial é importante para identificar características que deverão ser consideradas nas etapas posteriores de preparação dos dados e aplicação do algoritmo K-Means.

In [4]:
titanic = pd.read_csv('https://raw.githubusercontent.com/EMbeDS-education/IPDPP-GSSI-20202021/main/datasets/TITANIC.csv')

In [ ]:
titanic.head()

O conjunto de dados possui informações relacionadas aos passageiros do Titanic, incluindo características pessoais, informações da viagem e dados relacionados à sobrevivência. A seguir, são apresentadas brevemente as variáveis presentes no conjunto de dados.
| Variável    | Descrição                                                                       |
| ----------- | ------------------------------------------------------------------------------- |
| `pclass`    | Classe em que o passageiro estava viajando.                                     |
| `survived`  | Indica se o passageiro sobreviveu ao naufrágio.                                 |
| `Residence` | Código relacionado ao local de residência do passageiro.                        |
| `name`      | Nome do passageiro.                                                             |
| `age`       | Idade do passageiro.                                                            |
| `sibsp`     | Quantidade de irmãos ou cônjuges que estavam a bordo.                           |
| `parch`     | Quantidade de pais ou filhos que estavam a bordo.                               |
| `ticket`    | Número ou identificação do bilhete do passageiro.                               |
| `fare`      | Valor pago pelo passageiro pela passagem.                                       |
| `cabin`     | Identificação da cabine do passageiro.                                          |
| `embarked`  | Porto em que o passageiro embarcou.                                             |
| `boat`      | Identificação do bote salva-vidas utilizado pelo passageiro, quando disponível. |
| `body`      | Identificação associada ao corpo do passageiro, quando disponível.              |
| `home.dest` | Local de origem ou destino do passageiro.                                       |
| `Gender`    | Gênero do passageiro.                                                           |


Das variaveis do conjunto de dados as que chamam atenção são: <br>
```age``` que está como ```str```, mas representa idade; <br>
```fare``` que está como ```str```, mas representa uma tarifa numérica; <br>
```body``` também está como ```str```, embora represente um número/identificador.

In [ ]:
titanic.dtypes

A análise dos tipos de dados mostrou que o conjunto de dados possui variáveis numéricas e categóricas. Algumas variáveis que representam valores numéricos, como `age` e `fare`, estão armazenadas como texto (`str`) e poderão necessitar de conversão durante o pré-processamento.

Para verificar a existência de valores faltantes, foi utilizada a função `isnull().sum()`. O resultado mostrou que todas as variáveis apresentam zero valores ausentes identificados como `NaN`. Dessa forma, não foram identificados valores faltantes por esse método na etapa inicial de exploração dos dados.

In [ ]:
titanic.isnull().sum()

Para verificar a existência de registros duplicados, foi utilizada a função `duplicated().sum()`, que identificou **0 linhas duplicadas** no conjunto de dados. Dessa forma, não foi necessário remover nenhum registro por duplicidade.

In [11]:
titanic.duplicated().sum()

np.int64(0)

Inicialmente, a função `isnull().sum()` não identificou valores ausentes no conjunto de dados. Entretanto, durante uma análise mais detalhada, verificou-se que algumas variáveis possuíam valores faltantes representados como strings vazias. Dessa forma, foi necessário realizar uma verificação adicional para identificar essas ocorrências.

Foram identificados valores faltantes nas variáveis `age` (263), `fare` (1), `cabin` (1014), `embarked` (2), `boat` (823), `body` (1188) e `home.dest` (564).


In [ ]:
valoresFaltantes = ["?", "NA", "NaN", "nan", "NAN", "n/a", "N/A", "null", "NULL", "Null", " "]
for coluna in titanic.select_dtypes(include="str").columns:
    encontrados = titanic[coluna][titanic[coluna].isin(valoresFaltantes)].value_counts()

    if not encontrados.empty:
        print(f"\n{coluna}:")
        print(encontrados)

Para o tratamento dos valores faltantes, inicialmente foram consideradas as características e a quantidade de ausências de cada variável. Na variável age, os 263 valores faltantes foram substituídos pela mediana, evitando a remoção de uma quantidade significativa de registros. Na variável fare, que apresentava apenas um valor faltante, também foi utilizada a mediana.

Para a variável categórica embarked, que apresentava dois valores faltantes, foi utilizada a moda como estratégia de preenchimento. Já as variáveis cabin, boat, body e home.dest apresentaram uma quantidade elevada de valores faltantes e, por isso, não foram consideradas adequadas para utilização no agrupamento. Além disso, boat e body possuem informações relacionadas às consequências do naufrágio, podendo introduzir informações associadas à variável de sobrevivência no agrupamento. Dessa forma, essas variáveis serão desconsideradas na seleção dos atributos utilizados pelo K-Means.

In [15]:
titanic["age"] = titanic["age"].replace(" ", np.nan)
titanic["age"] = pd.to_numeric(titanic["age"])

titanic["age"] = titanic["age"].fillna(titanic["age"].median())

In [17]:
titanic["fare"] = titanic["fare"].replace(" ", np.nan)
titanic["fare"] = pd.to_numeric(titanic["fare"])

titanic["fare"] = titanic["fare"].fillna(titanic["fare"].median())

In [18]:
titanic["embarked"] = titanic["embarked"].replace(" ", np.nan)

titanic["embarked"] = titanic["embarked"].fillna(
    titanic["embarked"].mode()[0]
)